# ROGII public h-blend fast CSV

In [ ]:

from pathlib import Path
import json
import numpy as np
import pandas as pd

COMP = Path('/kaggle/input/competitions/rogii-wellbore-geology-prediction')
if not COMP.exists():
    COMP = Path('/kaggle/input/rogii-wellbore-geology-prediction')
SRC = Path('/kaggle/input/datasets/nina2025/rogii-03')
if not SRC.exists():
    SRC = Path('/kaggle/input/rogii-03')
OUT = Path('/kaggle/working')
OUT.mkdir(parents=True, exist_ok=True)

params = {
    'id': 'id',
    'target': 'tvt',
    'type_sort': ['asc/desc', 0.30, 0.70],
    'subwts': [0.10, -0.03, -0.07],
    'subm': [
        {'name': '9.537', 'weight': 0.81},
        {'name': '9.765', 'weight': 0.15},
        {'name': '9.956', 'weight': 0.04},
    ],
}

print(f'COMP={COMP} exists={COMP.exists()}')
print(f'SRC={SRC} exists={SRC.exists()}')
print('SRC files=', sorted(p.name for p in SRC.glob('*.csv')))

def read_component(name: str) -> pd.DataFrame:
    p = SRC / f'{name}.csv'
    df = pd.read_csv(p)
    value_col = 'tvt' if 'tvt' in df.columns else ('target' if 'target' in df.columns else 'pred')
    return df[['id', value_col]].rename(columns={value_col: name})

parts = [read_component(s['name']) for s in params['subm']]
merged = parts[0]
for part in parts[1:]:
    merged = merged.merge(part, on='id', how='inner')
cols = [s['name'] for s in params['subm']]
main_weights = {s['name']: float(s['weight']) for s in params['subm']}
subwts = list(map(float, params['subwts']))
asc_weight = float(params['type_sort'][1])
desc_weight = float(params['type_sort'][2])

def directional_blend(row, descending: bool) -> float:
    ordered = sorted(cols, key=lambda c: row[c], reverse=descending)
    rank = {c: i for i, c in enumerate(ordered)}
    return float(sum(row[c] * (main_weights[c] + subwts[rank[c]]) for c in cols))

asc = merged.apply(lambda r: directional_blend(r, False), axis=1)
desc = merged.apply(lambda r: directional_blend(r, True), axis=1)
sub = merged[['id']].copy()
sub['tvt'] = desc_weight * desc + asc_weight * asc

sample = pd.read_csv(COMP / 'sample_submission.csv')
sub = sample[['id']].merge(sub, on='id', how='left')
missing = int(sub['tvt'].isna().sum())
if missing:
    fallback = float(np.nanmean([pd.read_csv(SRC / f'{c}.csv')['tvt'].mean() for c in cols]))
    sub['tvt'] = sub['tvt'].fillna(fallback)

report = {
    'route': 'nina_public_hblend_fast_csv',
    'rows': int(len(sub)),
    'expected_rows': int(len(sample)),
    'ids_exact': bool(sub['id'].astype(str).equals(sample['id'].astype(str))),
    'missing': missing,
    'min': float(sub['tvt'].min()),
    'max': float(sub['tvt'].max()),
    'mean': float(sub['tvt'].mean()),
    'components': cols,
    'main_weights': main_weights,
    'subwts': subwts,
    'asc_weight': asc_weight,
    'desc_weight': desc_weight,
}
assert report['rows'] == report['expected_rows'], report
assert report['ids_exact'], report
assert np.isfinite(sub['tvt'].to_numpy(float)).all(), report
sub[['id', 'tvt']].to_csv(OUT / 'submission.csv', index=False)
(OUT / 'blend_report.json').write_text(json.dumps(report, indent=2) + '\n')
print(json.dumps(report, indent=2))
